# 02 — Frozen municipality TWFE result

This notebook documents the **executed v1 municipality model**. The authoritative estimation path is `scripts/run_municipality_twfe.py` plus the GitHub Actions workflow. This notebook reads the committed result summary rather than re-estimating a different specification.

The final model is

\[
\log(R_{it}/Y_{it}) = \alpha_i + \delta_t + \beta\log(TI_{it}) + \varepsilon_{it},
\]

with municipality and year fixed effects and municipality-clustered standard errors. It is a descriptive within-municipality association, not a causal treatment effect.

In [ ]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS = ROOT / "results" / "processed"

result = json.loads((RESULTS / "municipality_twfe_results.json").read_text(encoding="utf-8"))
provenance = json.loads((RESULTS / "municipality_twfe_provenance.json").read_text(encoding="utf-8"))
result["design"], provenance

## Primary and balanced estimates

The unbalanced sample is primary because it preserves all complete municipality-years for municipalities observed in at least two years. The all-three-year balanced panel is a prespecified sensitivity.

In [ ]:
rows = []
for sample_name in ["primary_unbalanced", "balanced_sensitivity"]:
    for outcome in ["affordability", "rent", "income"]:
        item = result[sample_name][outcome]
        rows.append({
            "sample": sample_name,
            "outcome": outcome,
            "coefficient": item["coefficient"],
            "clustered_se": item["std_error_clustered"],
            "ci_95_low": item["ci_95_low"],
            "ci_95_high": item["ci_95_high"],
            "p_value": item["p_value"],
            "municipalities": item["municipalities"],
            "observations": item["n_observations"],
        })

pd.DataFrame(rows)

## Interpretation

The primary affordability coefficient is **0.03562** with a 95% confidence interval of **[-0.01643, 0.08767]** and **p = 0.180**. The balanced sensitivity is **0.03927** with a 95% confidence interval of **[-0.02577, 0.10430]** and **p = 0.237**.

The direction and magnitude are therefore stable across the two samples, but the uncertainty intervals include zero. The v1 conclusion is a modest positive estimated association that is statistically uncertain.

## Rent versus income decomposition

Because the same rows and fixed effects are used for all three equations, the coefficient identity follows from

\[\log(R/Y)=\log R-\log Y.\]


In [ ]:
primary = result["primary_unbalanced"]
identity = (
    primary["affordability"]["coefficient"]
    - (primary["rent"]["coefficient"] - primary["income"]["coefficient"])
)
{
    "rent_coefficient": primary["rent"]["coefficient"],
    "income_coefficient": primary["income"]["coefficient"],
    "affordability_coefficient": primary["affordability"]["coefficient"],
    "identity_gap": identity,
}

## Evidence boundary

The estimated affordability association is driven almost entirely by rent: the primary rent coefficient is 0.03443 while the income coefficient is -0.00119. None of these results identifies a causal tourism effect.

The originally proposed annual THCR and EHPI analyses are intentionally absent from v1 because historical active STR stock and post-2022 housing-stock support are insufficient. A causal extension would require a separate identification design, for example a regulation-based event study.